# 🛰️ GeoFM: Early Deforestation Detection & EUDR Compliance Suite
### *High-Performance Geospatial AI Training Pipeline with LoRA PEFT & GPU Acceleration*

This notebook trains the **GeoFM Early Deforestation Detection** neural architectures using Google Colab’s NVIDIA GPU acceleration.

#### 🔬 Architectural Capabilities:
- **Flexible Model Scale**: Select between Siamese Bitemporal Network, U-Net with ViT bottleneck, or Multimodal (Optical + SAR) Cross-Attention.
- **Parameter-Efficient Fine-Tuning (LoRA PEFT)**: Freezes >99% of backbone weights, training low-rank projection adapters ($W_q, W_v$) for fast convergence.
- **Class-Imbalanced Anomaly Loss**: Compound Focal-Tversky loss tuned for rare canopy loss anomalies ($<5\%$ landscape area).
- **Spatial Block Cross-Validation**: Prevents spatial autocorrelation data leakage using geographic buffer margins (TorchGeo style).
- **Offline Chip Caching**: Standardized `.npz` chip serialization for maximum data throughput.


---
## 1. Hardware Accelerator Verification
Verify that the NVIDIA GPU (Tesla T4, V100, or A100) is allocated and ready.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    '❌ CUDA is not active! In Colab menu, go to: \n'
    'Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save.'
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'✓ GPU Active:    {gpu_name}')
print(f'✓ Total VRAM:    {vram_gb:.2f} GB')
print(f'✓ PyTorch:       {torch.__version__}')
print(f'✓ cuDNN Version: {torch.backends.cudnn.version()}')


---
## 2. Clone Repository & Install Dependencies
Clone the `safiydev/geofm-deforestation` repository and install dependencies.


In [ ]:
import os
from getpass import getpass

# Retrieve GitHub Personal Access Token (PAT) if repo is private
github_token = os.environ.get('GITHUB_TOKEN') or getpass('Enter GitHub Personal Access Token (or press Enter if public): ')

if github_token:
    repo_url = f'https://{github_token}@github.com/safiydev/geofm-deforestation.git'
else:
    repo_url = 'https://github.com/safiydev/geofm-deforestation.git'

# Clean previous checkout if restarting
!rm -rf geofm-deforestation
!git clone {repo_url}
%cd geofm-deforestation

# Install dependencies and local editable package
!pip install -q rasterio pystac fastapi uvicorn matplotlib pandas
!pip install -q -e .

# Run test suite to verify all modules
!pytest tests/ -q


---
## 3. Dataset Scale & Model Configuration

### 💡 Guidance on Chips, Scale, and Training Time:

| Scale Option | Chip Count ($256 \times 256$) | Compute Time (Colab T4) | Best Used For |
|---|---|---|---|
| **Quick Smoke Test** | **40 chips** | **~1 minute** | Verifying forward/backward pass, loss computation, and checkpoint saving. |
| **Regional Pilot** | **250 chips** | **~5 – 8 minutes** | High-precision baseline for a specific municipality, forest reserve, or supply shed. |
| **Production Baseline** | **1,000 chips** | **~20 – 30 minutes** | Multi-seasonal tropical dataset with wet/dry season phenological diversity. |

### 🧠 Model Architecture Options:
- `siamese` (**Recommended**): Dual-branch shared-weight encoder taking pre-event ($T_1$) and post-event ($T_2$) scenes, extracting multi-scale differential features.
- `unet`: U-Net semantic segmentation with Vision Transformer (ViT) global attention bottleneck for single-scene canopy gap delineation.
- `multimodal`: Optical (Sentinel-2) + SAR (Sentinel-1) hybrid cross-attention with dynamic cloud gating (essential for persistent cloud regions).


In [ ]:
# @title 🛠️ Interactive Training Configuration { run: 'auto' }

DATASET_SCALE = 'Regional Pilot (250 chips)' #@param ['Quick Smoke Test (40 chips)', 'Regional Pilot (250 chips)', 'Production Baseline (1000 chips)']
MODEL_ARCHITECTURE = 'siamese' #@param ['siamese', 'unet', 'multimodal']
USE_LORA = True #@param {type:'boolean'}
LORA_RANK = 8 #@param [4, 8, 16, 32] {type:'raw'}
LORA_ALPHA = 16.0 #@param {type:'number'}
LOSS_TYPE = 'compound' #@param ['compound', 'focal', 'tversky']
EPOCHS = 25 #@param {type:'integer'}
BATCH_SIZE = 8 #@param [2, 4, 8, 16, 32] {type:'raw'}
LEARNING_RATE = 0.001 #@param {type:'number'}
CHIP_SIZE = 256 #@param [128, 256, 512] {type:'raw'}

# Map dataset scale selection to integer chip counts
scale_map = {
    'Quick Smoke Test (40 chips)': 40,
    'Regional Pilot (250 chips)': 250,
    'Production Baseline (1000 chips)': 1000,
}
NUM_CHIPS = scale_map[DATASET_SCALE]
print(f'✓ Configuration Locked:')
print(f'  • Chips:        {NUM_CHIPS} paired tiles ({CHIP_SIZE}x{CHIP_SIZE} px)')
print(f'  • Architecture: {MODEL_ARCHITECTURE}')
print(f'  • LoRA PEFT:    {USE_LORA} (Rank={LORA_RANK}, Alpha={LORA_ALPHA})')
print(f'  • Training:     {EPOCHS} Epochs | Batch Size: {BATCH_SIZE} | LR: {LEARNING_RATE}')


---
## 4. Geospatial Data Ingestion & Spatial Block Cross-Validation
In Earth Observation, neighboring pixels exhibit extreme spatial autocorrelation. Random splitting causes severe data leakage and artificially inflated accuracy.

We use **Spatial Block Cross-Validation** with a safety buffer margin: candidate training chips located within the buffer distance of validation blocks are withheld.


In [ ]:
import os
import numpy as np
from geofm.utils.synthetic_data import generate_synthetic_dataset
from geofm.training.dataset import (
    cache_dataset_to_npz,
    load_dataset_from_npz,
    SpatialBlockSplitter,
    GeospatialDeforestationDataset
)

# 1. Generate or prepare high-fidelity geospatial paired chips
print(f'Generating {NUM_CHIPS} paired satellite chips ({CHIP_SIZE}x{CHIP_SIZE} px)...')
chips = generate_synthetic_dataset(num_samples=NUM_CHIPS, height=CHIP_SIZE, width=CHIP_SIZE, coord_step=0.04)

# 2. Cache chips locally as .npz files for high-speed offline GPU streaming
cache_dir = './data/chips_cache'
saved_files = cache_dataset_to_npz(chips, output_dir=cache_dir)
print(f'✓ Cached {len(saved_files)} chips to disk: {cache_dir}')

# 3. Load chips and apply Spatial Block Cross-Validation
loaded_chips = load_dataset_from_npz(cache_dir)
splitter = SpatialBlockSplitter(
    grid_size_deg=0.1,
    buffer_size_deg=0.02,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42
)
train_idx, val_idx, test_idx = splitter.split(loaded_chips)
print(f'✓ Spatially Disjoint Split: {len(train_idx)} Train | {len(val_idx)} Validation | {len(test_idx)} Test chips')


---
## 5. Model Training Execution
Launch the training loop. Notice how LoRA adapts the query and value matrices ($W_q, W_v$) while keeping base layers frozen.


In [ ]:
lora_flag = '--use-lora' if USE_LORA else ''

!python geofm/train_cli.py \
    --model-type {MODEL_ARCHITECTURE} \
    {lora_flag} \
    --lora-rank {LORA_RANK} \
    --lora-alpha {LORA_ALPHA} \
    --loss {LOSS_TYPE} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --checkpoint-dir ./checkpoints


---
## 6. Training & Validation Convergence Curves
Plot the historical loss, IoU, and F1 trajectories across all epochs.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = './checkpoints/metrics_history.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # 1. Loss trajectory
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#d90429', lw=2.5)
    axes[0].plot(df['epoch'], df['val_loss'], label='Val Loss', color='#f77f00', lw=2.5, linestyle='--')
    axes[0].set_title('Compound Focal-Tversky Loss Trajectory', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Loss', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # 2. Detection accuracy (IoU & F1)
    axes[1].plot(df['epoch'], df['val_iou'] * 100, label='Validation IoU (%)', color='#2d6a4f', lw=2.5)
    if 'val_f1' in df.columns:
        axes[1].plot(df['epoch'], df['val_f1'] * 100, label='Validation F1 (%)', color='#1b4332', lw=2.5, linestyle='--')
    axes[1].set_title('Deforestation IoU & F1 Convergence', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=10)
    axes[1].set_ylabel('Score (%)', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    
    best_iou = df['val_iou'].max() * 100
    print(f'🏆 Highest Achieved Validation IoU: {best_iou:.2f}%')
else:
    print('Metrics CSV not found. Ensure training completed successfully.')


---
## 7. Interactive Prediction & Alert Polygonization
Run inference on a validation chip and convert the raw neural probabilities into clean, MMU-filtered vector polygons.


In [ ]:
import torch
from geofm.postprocessing.polygonizer import Polygonizer
from geofm.preprocessing.cog_reader import GeoTransform

# Load sample validation chip
val_sample = loaded_chips[val_idx[0]]
t1_tensor = torch.from_numpy(val_sample.t1_optical).unsqueeze(0).cuda().float()
t2_tensor = torch.from_numpy(val_sample.t2_optical).unsqueeze(0).cuda().float()

# Load trained best model
model_chk = torch.load('./checkpoints/best_model.pt', map_location='cuda')
from geofm.models.siamese import SiameseChangeDetector
from geofm.models.lora import apply_lora_to_model

model = SiameseChangeDetector(in_channels=4)
if USE_LORA:
    model, _ = apply_lora_to_model(model, rank=LORA_RANK, alpha=LORA_ALPHA)
model.load_state_dict(model_chk['model_state_dict'], strict=False)
model.cuda().eval()

with torch.no_grad():
    probs = torch.sigmoid(model(t1_tensor, t2_tensor)).squeeze().cpu().numpy()

# Polygonize using Minimum Mapping Unit (MMU = 0.10 ha)
geotransform = GeoTransform(west=val_sample.lon, north=val_sample.lat, x_size_deg=0.0001, y_size_deg=0.0001)
polygonizer = Polygonizer(threshold=0.5, mmu_hectares=0.10, geotransform=geotransform)
alerts_fc = polygonizer.polygonize(probs)

print(f'✓ Extracted {len(alerts_fc["features"])} Legal Deforestation Alert Polygons from Chip!')
for f in alerts_fc['features'][:3]:
    p = f['properties']
    print(f'  - Alert {p["cluster_id"]}: Area = {p["area_ha"]} ha | Level: {p["alert_level"]} | Centroid: {p["centroid"]}')


---
## 8. Export & Download Model Checkpoints
Download `best_model.pt` to deploy to your tablet, server, or cloud API.


In [ ]:
from google.colab import files

if os.path.exists('./checkpoints/best_model.pt'):
    files.download('./checkpoints/best_model.pt')
    print('✓ Triggered download for best_model.pt')

if os.path.exists('./checkpoints/metrics_history.csv'):
    files.download('./checkpoints/metrics_history.csv')
    print('✓ Triggered download for metrics_history.csv')
